# wrap-forward-fn-generic — worked example 2: wrap_forward_fn with kwargs and is_differentiable

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `wrap-forward-fn-generic`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

The full factory threads `kwargs` through to both the forward call and the stored Recipe, and respects an `is_differentiable` flag. `requires_grad` is true only when the op is differentiable AND some input requires grad; only then is a Recipe (capturing func, raw args, kwargs, and parents) attached for the backward pass.

## Worked solution

We extend the wrapper to record a Recipe.

1. Unbox and call as before, with `kwargs` passed through to `fwd_fn`.
2. We compute `requires_grad = is_differentiable and any(input is a Tensor with requires_grad)`.
3. We box the result with that flag. If `requires_grad`, we attach a `Recipe(fwd_fn, raw_args, kwargs, parents)` where `parents` maps arg index → Tensor for each tensor input — the backward pass needs both the kwargs (e.g. `dim=`) and the parent links.
4. Non-differentiable ops skip the Recipe entirely.

We wrap `t.sum` with a `dim=` kwarg over a grad-requiring tensor and print that a Recipe was attached and stored the kwarg.

In [ ]:
import torch as t
from dataclasses import dataclass

@dataclass
class Recipe:
    func: object
    args: tuple
    kwargs: dict
    parents: dict

class Tensor:
    def __init__(self, array, requires_grad: bool = False):
        self.array = array if isinstance(array, t.Tensor) else t.tensor(array)
        self.requires_grad = requires_grad
        self.recipe = None

def wrap_forward_fn(fwd_fn, is_differentiable: bool = True):
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, Tensor) else a for a in args)
        out_raw = fwd_fn(*raw_args, **kwargs)
        requires_grad = is_differentiable and any(
            isinstance(a, Tensor) and a.requires_grad for a in args
        )
        out = Tensor(out_raw, requires_grad)
        if requires_grad:
            parents = {i: a for i, a in enumerate(args) if isinstance(a, Tensor)}
            out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return tensor_func

sum_fn = wrap_forward_fn(t.sum)
x = Tensor([[1.0, 2.0], [3.0, 4.0]], requires_grad=True)
out = sum_fn(x, dim=0)
print('requires_grad:', out.requires_grad)
print('recipe kwargs:', out.recipe.kwargs)
print('values:', out.array.tolist())